# Convert Geojson to CSV for easier manual additions

This notebooks loads in the merged geojson produced in [01_geojson_merge.ipynb](01_geojson_merge.ipynb) and saves it out as a CSV to be manually modified and added to.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

In [ ]:
import subprocess
import sys


In [ ]:
file = "v5.geojson"

with open(file) as f:
    geojson = json.load(f)

# Extract properties from features
data = []
for feature in geojson['features']:
    row = feature['properties'].copy()
    row['geometry_type'] = feature['geometry']['type']
    row['coordinates'] = feature['geometry']['coordinates']
    data.append(row)

df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
# df.to_csv("temp_output_file.tsv", sep="\t")

In [ ]:
df.drop(columns=['visibility'], inplace=True)

In [ ]:
df

In [ ]:
df = df.explode('related_artefacts')


In [ ]:
df

In [ ]:
# df.to_csv("extended_temp_output_file.tsv", sep="\t")

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
import time

def fetch_iiif_manifest(digital_library_url):
    """
    Fetch IIIF manifest URL from a Leeds digital library item page.
    
    Process:
    1. Fetch the digital library item page
    2. Find the link to "View more information about this item" (special-collections-explore)
    3. Fetch that page and extract the IIIF manifest URL
    
    Args:
        digital_library_url: URL like https://digital.library.leeds.ac.uk/15933/
    
    Returns:
        Manifest URL or None if not found
    """

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Connection': 'keep-alive',
    }
    
    try:
        # Step 1: Fetch initial digital library page
        response = requests.get(digital_library_url, timeout=10, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Step 2: Find the "View more information about this item" link
        more_info_link = None
        for link in soup.find_all('a', href=True):
            if 'special-collections-explore' in link['href']:
                more_info_link = link['href']
                break
        
        if not more_info_link:
            print(f"Could not find special-collections-explore link")
            return None
        
        # Ensure absolute URL
        if not more_info_link.startswith('http'):
            more_info_link = urljoin(digital_library_url, more_info_link)
        
        # Step 3: Fetch the special collections page with referer and additional headers
        headers_with_referer = headers.copy()
        headers_with_referer['Referer'] = digital_library_url
        
        response = requests.get(more_info_link, timeout=10, headers=headers_with_referer)
        
        # Don't raise on 403
        if response.status_code == 403:
            print(f"403 Forbidden")
        elif response.status_code != 200:
            response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Step 4: Extract the IIIF manifest URL
        # Look for the manifest link (usually contains iiif.library.leeds.ac.uk)
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'iiif.library.leeds.ac.uk' in href and 'presentation' in href:
                return href
        
        # Fallback: look in text for manifest URL pattern
        page_text = soup.get_text()
        match = re.search(r'https://iiif\.library\.leeds\.ac\.uk/presentation/[^\s\)\"\'<]+', page_text)
        if match:
            return match.group(0)
        
        return None
    except requests.exceptions.RequestException as e:
        print(f"Request Error: {e}")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

# Test with one of your URLs
test_url = "https://digital.library.leeds.ac.uk/15945/"
print(f"Testing: {test_url}")
manifest = fetch_iiif_manifest(test_url)
print(f"Manifest: {manifest}")

In [ ]:
import time

def add_manifests_to_dataframe(df, url_column='related_artefacts', delay=1):
    """
    Add manifest URLs to dataframe by fetching from digital library pages.
    Includes delay to avoid overwhelming the server.
    
    Args:
        df: DataFrame with digital library URLs
        url_column: Column name containing the URLs
        delay: Seconds to wait between requests (default: 1)
    
    Returns:
        DataFrame with new 'manifest' column
    """
    df = df.copy()
    manifests = []
    
    for idx, url in enumerate(df[url_column]):
        print(f"[{idx+1}/{len(df)}] Fetching {url}...", end=" ")
        manifest = fetch_iiif_manifest(url)
        manifests.append(manifest)
        print(f"yes {manifest if manifest else 'no'}")
        time.sleep(delay)
    
    df['manifest'] = manifests
    return df



In [ ]:
df_with_manifests = add_manifests_to_dataframe(df)
# df_with_manifests.to_csv("extended_temp_output_file_with_manifests.tsv", sep="\t")

In [ ]:
df_with_manifests

In [ ]:
# load in table of URLs to check which we're missing

compare_table = pd.read_csv("Bingley_collection_gc_urls.csv")

In [ ]:
compare_table

In [ ]:
# Find URLs in compare_table that are NOT in df_with_manifests
urls_in_compare = set(compare_table['url'].dropna())
urls_in_df = set(df_with_manifests['related_artefacts'].dropna())

missing_urls = urls_in_compare - urls_in_df

print(f"URLs in compare_table: {len(urls_in_compare)}")
print(f"URLs in df_with_manifests: {len(urls_in_df)}")
print(f"Missing URLs: {len(missing_urls)}")

# Create new dataframe with missing URLs
missing_df = compare_table[compare_table['url'].isin(missing_urls)].copy()

print(f"\nMissing URLs dataframe shape: {missing_df.shape}")
missing_df

In [ ]:
# Fetch manifests for missing URLs and add to df_with_manifests
print(f"Fetching {len(missing_df)} missing URLs...\n")

new_rows = []
for idx, row in missing_df.iterrows():
    url = row['url']
    print(f"[{len(new_rows)+1}/{len(missing_df)}] Fetching {url}...", end=" ")
    manifest = fetch_iiif_manifest(url)
    print(f"Yes {manifest if manifest else 'no'}")
    
    # Create new row with url as related_artefacts, manifest, and placeholder values
    new_row = {col: None for col in df_with_manifests.columns}
    new_row['related_artefacts'] = url
    new_row['manifest'] = manifest
    new_rows.append(new_row)
    time.sleep(0.5)  # Shorter delay between requests

# Create new dataframe from new rows
new_rows_df = pd.DataFrame(new_rows)

# Append to df_with_manifests
df_with_manifests_updated = pd.concat([df_with_manifests, new_rows_df], ignore_index=True)




In [ ]:
df_with_manifests_updated

In [ ]:
df_with_manifests_updated.to_csv("extended_temp_output_file_with_manifests.tsv", sep="\t")